# 第 8 章 乱数とシミュレーション: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/economics-programming/08-random-numbers-and-simulation.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
seed = 0           # 乱数の種（同じ値なら、同じ結果になる）
price = 10         # 需要を観測する価格
intercept = 100    # 需要 Q = a - bP の切片 a
slope = 2          # 需要の傾き b
sigma = 10         # 観測値のばらつき（標準偏差）
n = 1000           # 観測値の個数
sample_size = 5    # サイコロを振る回数（1 回の試行あたり）
trials = 1000      # 試行の回数
bins = 30          # ヒストグラムの棒の数

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

## 大数の法則: 観測値の累積平均

In [ ]:
import random
import statistics

from lib.chapter08.simulation import running_means, sample_means, simulate_demand

rng = random.Random(seed)
observations = simulate_demand(price, intercept, slope, sigma, n, rng)
means = running_means(observations)
print(f"累積平均: 1 個 {means[0]:.2f}、10 個 {means[min(9, n - 1)]:.2f}、"
      f"全部 {means[-1]:.2f}（真の値 {intercept - slope * price}）")

## 図: 観測値の分布

In [ ]:
from lib.chapter08.simulation import plot_histogram

plot_histogram(observations, bins)

## 中心極限定理: サイコロの平均

In [ ]:
averages = sample_means(sample_size, trials, random.Random(seed))
print(f"平均の平均 {statistics.mean(averages):.3f}、"
      f"標準偏差 {statistics.stdev(averages):.3f}")

In [ ]:
plot_histogram(averages, bins)

## 試してみよう

- `sigma` を 0 にすると、観測値はすべて真の値 80 で、累積平均は最初から 80 です。
  30 にすると、観測値のばらつきが大きく、最小が約 -7、最大が約 187 になります
  （それでも、累積平均は、n = 1000 で約 80.5 に近づきます）
- `sample_size` を 1 にすると、平均の標準偏差は 1.74、5 で 0.77、30 で 0.31 に
  なります（平均は、どれも約 3.5 で、標準偏差は、回数の平方根に反比例して、
  小さくなります）
- 図の形は、`sample_size` が大きいほど、釣り鐘型（正規分布）に近づきます